In [17]:
# ============================================================
# AMRIS — ML VOLATILITY EXPERIMENTS
# Environment and reproducibility
# ============================================================

from pathlib import Path
import sys
import platform
import warnings

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()

# Locate the project root robustly if Jupyter starts elsewhere.
if not (PROJECT_ROOT / "data" / "research_outputs" / "v1").exists():
    candidates = [
        Path(r"D:\DS Projects\Adaptive Market Risk Intelligence System"),
        PROJECT_ROOT.parent,
    ]

    PROJECT_ROOT = next(
        (
            p.resolve()
            for p in candidates
            if (p / "data" / "research_outputs" / "v1").exists()
        ),
        PROJECT_ROOT,
    )

RESEARCH_DIR = PROJECT_ROOT / "data" / "research_outputs" / "v1"
ML_OUTPUT_DIR = PROJECT_ROOT / "data" / "ml_experiments" / "v1"

FORECAST_FILE = RESEARCH_DIR / "oos_forecasts.csv"
PERFORMANCE_FILE = RESEARCH_DIR / "model_performance.csv"

assert FORECAST_FILE.is_file(), f"Missing file: {FORECAST_FILE}"
assert PERFORMANCE_FILE.is_file(), f"Missing file: {PERFORMANCE_FILE}"
assert ML_OUTPUT_DIR.is_dir(), f"Missing directory: {ML_OUTPUT_DIR}"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project root:", PROJECT_ROOT)
print("Frozen research directory:", RESEARCH_DIR)
print("New ML output directory:", ML_OUTPUT_DIR)
print("Random seed:", SEED)
print("Environment setup: PASS")

Python: 3.13.7
Platform: Windows-11-10.0.26200-SP0
Project root: D:\DS Projects\Adaptive Market Risk Intelligence System
Frozen research directory: D:\DS Projects\Adaptive Market Risk Intelligence System\data\research_outputs\v1
New ML output directory: D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1
Random seed: 42
Environment setup: PASS



## 1. Research Baseline and Data Contract

This notebook extends the existing Adaptive Market Risk Intelligence System
(AMRIS) with separate machine-learning experiments.

The existing out-of-sample forecasts and model-performance results are
read-only benchmarks. This notebook will not regenerate or overwrite them.

Before selecting features or defining a prediction target, we will inspect
the available columns, date range, observation count, missing values, and
duplicate dates.

**Research principle:** Features must reflect information available at the
forecast origin. Model comparisons must use aligned dates and a consistent
target definition.


In [18]:

# ============================================================
# Read-only loading and structural inspection
# ============================================================

forecasts = pd.read_csv(
    FORECAST_FILE,
    parse_dates=["Date"],
)

model_performance = pd.read_csv(PERFORMANCE_FILE)

# Structural checks
assert not forecasts.empty, "Frozen forecast dataset is empty."
assert "Date" in forecasts.columns, "Forecast dataset has no Date column."
assert forecasts["Date"].notna().all(), "Missing dates detected."
assert not forecasts["Date"].duplicated().any(), "Duplicate dates detected."
assert forecasts["Date"].is_monotonic_increasing, (
    "Forecast dates are not sorted chronologically."
)

print("=" * 65)
print("FROZEN OUT-OF-SAMPLE FORECASTS")
print("=" * 65)
print("Shape:", forecasts.shape)
print("Date range:", forecasts["Date"].min(), "to", forecasts["Date"].max())
print("\nColumns:")
for column in forecasts.columns:
    print(f"  - {column}")

print("\nData types:")
print(forecasts.dtypes.to_string())

print("\nMissing values:")
print(forecasts.isna().sum().to_string())

print("\nFirst 5 rows:")
display(forecasts.head())

print("\n" + "=" * 65)
print("FROZEN MODEL PERFORMANCE")
print("=" * 65)
print("Shape:", model_performance.shape)
print("\nColumns:")
for column in model_performance.columns:
    print(f"  - {column}")

display(model_performance.head(10))

print("\nRead-only dataset inspection: PASS")


FROZEN OUT-OF-SAMPLE FORECASTS
Shape: (673, 12)
Date range: 2024-01-01 00:00:00 to 2026-09-22 00:00:00

Columns:
  - Date
  - NIFTY_Log_Return
  - Actual_Volatility
  - India_VIX_PreReturn
  - Historical_20D
  - EWMA
  - GARCH_t
  - GJR_GARCH_t
  - EGARCH_t
  - Historical_20D_VIX
  - EWMA_VIX
  - EGARCH_t_VIX

Data types:
Date                   datetime64[us]
NIFTY_Log_Return              float64
Actual_Volatility             float64
India_VIX_PreReturn           float64
Historical_20D                float64
EWMA                          float64
GARCH_t                       float64
GJR_GARCH_t                   float64
EGARCH_t                      float64
Historical_20D_VIX            float64
EWMA_VIX                      float64
EGARCH_t_VIX                  float64

Missing values:
Date                   0
NIFTY_Log_Return       0
Actual_Volatility      0
India_VIX_PreReturn    0
Historical_20D         0
EWMA                   0
GARCH_t                0
GJR_GARCH_t            0
EGA

,Date,NIFTY_Log_Return,Actual_Volatility,India_VIX_PreReturn,Historical_20D,EWMA,GARCH_t,GJR_GARCH_t,EGARCH_t,Historical_20D_VIX,EWMA_VIX,EGARCH_t_VIX
0,2024-01-01,0.000483,0.000483,14.50,0.007280,0.011568,0.007547,0.006755,0.005899,0.005299,0.004533,0.004508
1,2024-01-02,-0.003506,0.003506,14.50,0.007284,0.011216,0.006990,0.006427,0.006179,0.005292,0.004597,0.004651
2,2024-01-03,-0.006875,0.006875,14.58,0.006246,0.010908,0.006801,0.006577,0.006909,0.005408,0.004719,0.005065
3,2024-01-04,0.006543,0.006543,14.10,0.006411,0.010709,0.007200,0.007510,0.006681,0.005085,0.004408,0.004787
4,2024-01-05,0.002407,0.002407,13.33,0.006488,0.010506,0.007104,0.007052,0.006374,0.004576,0.003881,0.004372



FROZEN MODEL PERFORMANCE
Shape: (8, 6)

Columns:
  - Model
  - MAE
  - RMSE
  - N
  - OOS_Start
  - OOS_End


,Model,MAE,RMSE,N,OOS_Start,OOS_End
0,Historical_20D,0.004923,0.006564,673,2024-01-01,2026-09-22
1,EWMA,0.004924,0.006419,673,2024-01-01,2026-09-22
2,GARCH_t,0.004852,0.006291,673,2024-01-01,2026-09-22
3,GJR_GARCH_t,0.004865,0.006349,673,2024-01-01,2026-09-22
4,EGARCH_t,0.004628,0.005796,673,2024-01-01,2026-09-22
5,Historical_20D_VIX,0.003863,0.005630,673,2024-01-01,2026-09-22
6,EWMA_VIX,0.003863,0.005637,673,2024-01-01,2026-09-22
7,EGARCH_t_VIX,0.003737,0.005262,673,2024-01-01,2026-09-22



Read-only dataset inspection: PASS



## 2. Target Definition and Leakage Prevention

### Prediction target

The initial ML experiment will predict the next trading day's absolute
NIFTY log return:

\[
Y_{t+1} = |r_{t+1}|
\]

where \(r_{t+1}\) is the next trading day's log return.

This is a daily realized-volatility proxy, not a direct observation of
latent volatility.

### Information available at forecast time

At the end of trading day \(t\), candidate features may include:

- Lagged absolute returns and squared returns
- Historical volatility calculated using past returns only
- EWMA volatility updated through day \(t\)
- India VIX observed by the forecast origin

The target is shifted forward by one row. Features must not include
the target day's return or any forecast derived from that return.

### Evaluation design

The ML model will be evaluated chronologically. Random train/test splitting
will not be used. The existing frozen statistical-model forecasts will
remain unchanged and serve as benchmarks on matching dates.


In [19]:

# ============================================================
# Locate canonical market inputs (read-only)
# ============================================================

DATA_DIR = PROJECT_ROOT / "data" / "processed"

candidate_files = [
    DATA_DIR / "nifty50_prices_clean.csv",
    DATA_DIR / "india_vix_clean_daily_2018-01-01_2026-09-22.csv",
]

for path in candidate_files:
    print(f"{path.name}: {'FOUND' if path.is_file() else 'NOT FOUND'}")

assert all(path.is_file() for path in candidate_files), (
    "A canonical market input file is missing. "
    "Do not substitute a different dataset silently."
)

nifty_raw = pd.read_csv(candidate_files[0])
vix_raw = pd.read_csv(candidate_files[1])

print("\nNIFTY source shape:", nifty_raw.shape)
print("NIFTY columns:", nifty_raw.columns.tolist())
display(nifty_raw.head(3))

print("\nIndia VIX source shape:", vix_raw.shape)
print("India VIX columns:", vix_raw.columns.tolist())
display(vix_raw.head(3))


nifty50_prices_clean.csv: FOUND
india_vix_clean_daily_2018-01-01_2026-09-22.csv: FOUND

NIFTY source shape: (2150, 6)
NIFTY columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


,Date,Open,High,Low,Close,Volume
0,2018-01-02,10477.549805,10495.200195,10404.650391,10442.200195,153400
1,2018-01-03,10482.650391,10503.599609,10429.549805,10443.200195,167300
2,2018-01-04,10469.400391,10513.000000,10441.450195,10504.799805,174900



India VIX source shape: (2135, 2)
India VIX columns: ['Date', 'India_VIX']


,Date,India_VIX
0,2018-01-01,13.35
1,2018-01-02,13.69
2,2018-01-03,13.62


In [20]:

# ============================================================
# Construct next-trading-day ML features
# ============================================================

# Parse and validate source dates.
nifty = nifty_raw.copy()
vix = vix_raw.copy()

nifty["Date"] = pd.to_datetime(nifty["Date"], errors="raise")
vix["Date"] = pd.to_datetime(vix["Date"], errors="raise")

nifty = nifty.sort_values("Date").reset_index(drop=True)
vix = vix.sort_values("Date").reset_index(drop=True)

assert nifty["Date"].notna().all()
assert vix["Date"].notna().all()
assert not nifty["Date"].duplicated().any()
assert not vix["Date"].duplicated().any()

# Compute daily log returns from canonical NIFTY closing prices.
nifty["Log_Return"] = np.log(nifty["Close"]).diff()
nifty["Abs_Return"] = nifty["Log_Return"].abs()

# Features use information available through the current NIFTY date.
nifty["Abs_Return_Lag1"] = nifty["Abs_Return"].shift(1)
nifty["Abs_Return_Lag2"] = nifty["Abs_Return"].shift(2)
nifty["Abs_Return_Lag3"] = nifty["Abs_Return"].shift(3)

nifty["Mean_Abs_Return_5D"] = (
    nifty["Abs_Return"].rolling(5, min_periods=5).mean()
)

nifty["Realized_Volatility_5D"] = (
    nifty["Log_Return"].rolling(5, min_periods=5).std(ddof=1)
)

nifty["Realized_Volatility_20D"] = (
    nifty["Log_Return"].rolling(20, min_periods=20).std(ddof=1)
)

# EWMA volatility uses returns through the current feature date.
nifty["EWMA_Volatility_20D"] = (
    nifty["Log_Return"]
    .ewm(span=20, adjust=False, min_periods=20)
    .std(bias=False)
)

# The VIX value must be dated on or before the feature date.
vix = vix.rename(columns={"India_VIX": "India_VIX_Feature"})

features = pd.merge_asof(
    nifty.sort_values("Date"),
    vix[["Date", "India_VIX_Feature"]].sort_values("Date"),
    on="Date",
    direction="backward",
    allow_exact_matches=True,
)

# Record the actual source date for the VIX observation.
vix_dates = vix[["Date"]].rename(columns={"Date": "VIX_Source_Date"})

features = pd.merge_asof(
    features.sort_values("Date"),
    vix_dates.sort_values("VIX_Source_Date"),
    left_on="Date",
    right_on="VIX_Source_Date",
    direction="backward",
    allow_exact_matches=True,
)

features["VIX_Change_1D"] = features["India_VIX_Feature"].pct_change()

# Target: absolute return on the next available NIFTY observation.
features["Target_Date"] = features["Date"].shift(-1)
features["Target_Abs_Return"] = features["Abs_Return"].shift(-1)

feature_columns = [
    "Abs_Return",
    "Abs_Return_Lag1",
    "Abs_Return_Lag2",
    "Abs_Return_Lag3",
    "Mean_Abs_Return_5D",
    "Realized_Volatility_5D",
    "Realized_Volatility_20D",
    "EWMA_Volatility_20D",
    "India_VIX_Feature",
    "VIX_Change_1D",
]

ml_dataset = features[
    ["Date", "Target_Date", "Target_Abs_Return"] + feature_columns
].dropna().reset_index(drop=True)

# Timing and numeric-integrity checks.
assert (ml_dataset["Date"] < ml_dataset["Target_Date"]).all()
assert np.isfinite(
    ml_dataset[feature_columns + ["Target_Abs_Return"]]
    .to_numpy(dtype=float)
).all()

assert (
    features["VIX_Source_Date"].dropna()
    <= features.loc[features["VIX_Source_Date"].notna(), "Date"]
).all()

print("Feature construction: PASS")
print("Feature rows:", len(ml_dataset))
print("Feature date range:", ml_dataset["Date"].min(), "to", ml_dataset["Date"].max())
print("Target date range:", ml_dataset["Target_Date"].min(), "to", ml_dataset["Target_Date"].max())
print("Number of features:", len(feature_columns))
print("Missing values in constructed dataset:", int(ml_dataset.isna().sum().sum()))

display(ml_dataset.head())


Feature construction: PASS
Feature rows: 2129
Feature date range: 2018-01-31 00:00:00 to 2026-09-21 00:00:00
Target date range: 2018-02-01 00:00:00 to 2026-09-22 00:00:00
Number of features: 10
Missing values in constructed dataset: 0


,Date,Target_Date,Target_Abs_Return,Abs_Return,Abs_Return_Lag1,Abs_Return_Lag2,Abs_Return_Lag3,Mean_Abs_Return_5D,Realized_Volatility_5D,Realized_Volatility_20D,EWMA_Volatility_20D,India_VIX_Feature,VIX_Change_1D
0,2018-01-31,2018-02-01,0.000980,0.001988,0.007281,0.005473,0.001476,0.003285,0.004583,0.004471,0.004877,15.930000,-0.029842
1,2018-02-01,2018-02-02,0.023539,0.000980,0.001988,0.007281,0.005473,0.003440,0.004534,0.004510,0.004703,14.110000,-0.114250
2,2018-02-02,2018-02-05,0.008779,0.023539,0.000980,0.001988,0.007281,0.007852,0.010973,0.007327,0.008809,15.250000,0.080794
3,2018-02-05,2018-02-06,0.015904,0.008779,0.023539,0.000980,0.001988,0.008513,0.009037,0.007590,0.008702,16.049999,0.052459
4,2018-02-06,2018-02-07,0.002055,0.015904,0.008779,0.023539,0.000980,0.010238,0.009552,0.008298,0.009326,20.020000,0.247352


In [21]:

# ============================================================
# Align ML targets with frozen OOS benchmarks
# ============================================================

benchmark = forecasts.copy()
benchmark["Date"] = pd.to_datetime(benchmark["Date"])

# Join each ML prediction origin to the frozen forecast for its target date.
aligned = ml_dataset.merge(
    benchmark,
    left_on="Target_Date",
    right_on="Date",
    how="inner",
    validate="many_to_one",
    suffixes=("_Feature", "_Benchmark"),
)

# Compare independently constructed target with frozen realized volatility.
aligned["Target_Absolute_Difference"] = (
    aligned["Target_Abs_Return"] - aligned["Actual_Volatility"]
).abs()

print("=" * 65)
print("ML / FROZEN BENCHMARK ALIGNMENT")
print("=" * 65)
print("ML rows before alignment:", len(ml_dataset))
print("Frozen benchmark rows:", len(benchmark))
print("Matched target dates:", len(aligned))
print("Unmatched ML target dates:", len(ml_dataset) - len(aligned))
print(
    "Maximum target difference:",
    f"{aligned['Target_Absolute_Difference'].max():.12g}"
)
print(
    "Mean target difference:",
    f"{aligned['Target_Absolute_Difference'].mean():.12g}"
)
print("Aligned target date range:",
      aligned["Target_Date"].min(), "to", aligned["Target_Date"].max())

display(
    aligned[
        [
            "Date_Feature",
            "Target_Date",
            "Target_Abs_Return",
            "Actual_Volatility",
            "EGARCH_t_VIX",
            "Target_Absolute_Difference",
        ]
    ].head(10)
)


ML / FROZEN BENCHMARK ALIGNMENT
ML rows before alignment: 2129
Frozen benchmark rows: 673
Matched target dates: 673
Unmatched ML target dates: 1456
Maximum target difference: 1.68138072909e-15
Mean target difference: 5.88061212268e-16
Aligned target date range: 2024-01-01 00:00:00 to 2026-09-22 00:00:00


,Date_Feature,Target_Date,Target_Abs_Return,Actual_Volatility,EGARCH_t_VIX,Target_Absolute_Difference
0,2023-12-29,2024-01-01,0.000483,0.000483,0.004508,8.449188e-16
1,2024-01-01,2024-01-02,0.003506,0.003506,0.004651,1.424208e-15
2,2024-01-02,2024-01-03,0.006875,0.006875,0.005065,1.589007e-15
3,2024-01-03,2024-01-04,0.006543,0.006543,0.004787,1.700029e-16
4,2024-01-04,2024-01-05,0.002407,0.002407,0.004372,1.071192e-16
5,2024-01-05,2024-01-08,0.009152,0.009152,0.004696,1.009609e-15
6,2024-01-08,2024-01-09,0.001479,0.001479,0.004769,3.606056e-16
7,2024-01-09,2024-01-10,0.003422,0.003422,0.004533,4.072263e-16
8,2024-01-10,2024-01-11,0.001317,0.001317,0.004258,1.081817e-15
9,2024-01-11,2024-01-12,0.011362,0.011362,0.004150,4.076600e-16



## 3. Chronological Evaluation Design

The frozen statistical forecasts cover 673 target dates from January 2024
through September 2026. They will be used as the common evaluation period.

ML training data must precede the evaluation period. Feature engineering
must be reproducible using information available at each forecast origin.

The first experiment will use a fixed, chronological training cutoff. The
ML models will be fitted without using evaluation-period targets.

Models will be compared using MAE and RMSE on identical target dates.
The frozen forecasts are reference benchmarks and will not be modified.


In [22]:

# ============================================================
# Chronological training/evaluation split
# ============================================================

EVALUATION_START = pd.Timestamp("2024-01-01")

# Keep only observations whose target date is within the frozen
# out-of-sample period.
evaluation_data = aligned.loc[
    aligned["Target_Date"] >= EVALUATION_START
].copy()

# Training rows must have targets strictly before the evaluation start.
training_data = ml_dataset.loc[
    ml_dataset["Target_Date"] < EVALUATION_START
].copy()

# Enforce the intended date boundary.
assert not training_data.empty, "No pre-evaluation training observations."
assert not evaluation_data.empty, "No aligned evaluation observations."
assert training_data["Target_Date"].max() < EVALUATION_START
assert evaluation_data["Target_Date"].min() >= EVALUATION_START
assert evaluation_data["Target_Date"].is_monotonic_increasing

print("Training observations:", len(training_data))
print(
    "Training target range:",
    training_data["Target_Date"].min(),
    "to",
    training_data["Target_Date"].max(),
)
print("Evaluation observations:", len(evaluation_data))
print(
    "Evaluation target range:",
    evaluation_data["Target_Date"].min(),
    "to",
    evaluation_data["Target_Date"].max(),
)
print("Chronological boundary checks: PASS")


Training observations: 1456
Training target range: 2018-02-01 00:00:00 to 2023-12-29 00:00:00
Evaluation observations: 673
Evaluation target range: 2024-01-01 00:00:00 to 2026-09-22 00:00:00
Chronological boundary checks: PASS



## 4. Initial Machine-Learning Models

We will evaluate three initial approaches:

1. **Median baseline:** Predicts the median absolute return observed in the training sample.
2. **Random Forest:** Captures nonlinear relationships and interactions among lagged market features.
3. **Histogram Gradient Boosting:** Provides a second tree-based method for comparison.

All models are fitted exclusively on pre-2024 training observations. Hyperparameters are specified in advance for this initial experiment; the evaluation period will not be used for model selection.

The models will be evaluated against the existing frozen statistical forecasts using MAE and RMSE on the same 673 target dates.


In [23]:

# ============================================================
# Fit initial ML models and compare with benchmarks
# ============================================================

from sklearn import __version__ as sklearn_version
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("scikit-learn version:", sklearn_version)

X_train = training_data[feature_columns].copy()
y_train = training_data["Target_Abs_Return"].copy()

X_eval = evaluation_data[feature_columns].copy()
y_eval = evaluation_data["Target_Abs_Return"].copy()

assert list(X_train.columns) == list(X_eval.columns)
assert len(X_eval) == len(y_eval) == 673
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_eval.to_numpy(dtype=float)).all()

models = {
    "ML_Median_Baseline": DummyRegressor(strategy="median"),
    "ML_Random_Forest": RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=10,
        max_features=0.8,
        random_state=SEED,
        n_jobs=-1,
    ),
    "ML_Hist_Gradient_Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=SEED,
    ),
}

ml_predictions = {}
result_rows = []

# Evaluate existing frozen statistical forecasts on these same target dates.
benchmark_models = [
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
    "Historical_20D_VIX",
    "EWMA_VIX",
    "EGARCH_t_VIX",
]

for model_name in benchmark_models:
    prediction = evaluation_data[model_name].to_numpy(dtype=float)

    result_rows.append({
        "Model": model_name,
        "Type": "Frozen statistical benchmark",
        "MAE": mean_absolute_error(y_eval, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_eval, prediction)),
        "N": len(y_eval),
    })

# Fit ML models using pre-evaluation observations only.
for model_name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_eval)

    assert len(prediction) == len(y_eval)
    assert np.isfinite(prediction).all()

    ml_predictions[model_name] = prediction

    result_rows.append({
        "Model": model_name,
        "Type": "Machine learning",
        "MAE": mean_absolute_error(y_eval, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_eval, prediction)),
        "N": len(y_eval),
    })

initial_results = (
    pd.DataFrame(result_rows)
    .sort_values(["RMSE", "MAE"])
    .reset_index(drop=True)
)

print("\nTraining observations:", len(X_train))
print("Evaluation observations:", len(X_eval))
print("Features:", len(feature_columns))
print("\nInitial model comparison — lower MAE/RMSE is better:")
display(initial_results.round(8))

print("\nTraining/evaluation separation: PASS")


scikit-learn version: 1.9.1

Training observations: 1456
Evaluation observations: 673
Features: 10

Initial model comparison — lower MAE/RMSE is better:


,Model,Type,MAE,RMSE,N
0,EGARCH_t_VIX,Frozen statistical benchmark,0.003737,0.005262,673
1,Historical_20D_VIX,Frozen statistical benchmark,0.003863,0.005630,673
2,EWMA_VIX,Frozen statistical benchmark,0.003863,0.005637,673
3,ML_Random_Forest,Machine learning,0.004013,0.005685,673
4,ML_Hist_Gradient_Boosting,Machine learning,0.004052,0.005767,673
5,EGARCH_t,Frozen statistical benchmark,0.004628,0.005796,673
6,ML_Median_Baseline,Machine learning,0.004075,0.005979,673
7,GARCH_t,Frozen statistical benchmark,0.004852,0.006291,673
8,GJR_GARCH_t,Frozen statistical benchmark,0.004865,0.006349,673
9,EWMA,Frozen statistical benchmark,0.004924,0.006419,673



Training/evaluation separation: PASS


In [24]:

# ============================================================
# Save initial ML experiment outputs
# Frozen research outputs remain read-only.
# ============================================================

from pathlib import Path

ML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save model comparison
results_file = ML_OUTPUT_DIR / "initial_model_comparison.csv"
initial_results.to_csv(results_file, index=False)

# Save actual values and all benchmark/ML predictions
prediction_output = pd.DataFrame({
    "Target_Date": evaluation_data["Target_Date"].to_numpy(),
    "Actual_Abs_Return": y_eval.to_numpy(),
})

for model_name in benchmark_models:
    prediction_output[model_name] = (
        evaluation_data[model_name].to_numpy(dtype=float)
    )

for model_name, prediction in ml_predictions.items():
    prediction_output[model_name] = prediction

assert len(prediction_output) == 673
assert prediction_output["Target_Date"].is_monotonic_increasing
assert prediction_output["Target_Date"].is_unique

prediction_file = ML_OUTPUT_DIR / "initial_model_predictions.csv"
prediction_output.to_csv(prediction_file, index=False)

print("Saved initial ML experiment outputs:")
print(results_file)
print(prediction_file)
print("\nObservations:", len(prediction_output))
print("Models compared:", len(initial_results))
print("Frozen research outputs modified: NO")


Saved initial ML experiment outputs:
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\initial_model_comparison.csv
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\initial_model_predictions.csv

Observations: 673
Models compared: 11
Frozen research outputs modified: NO


In [25]:

# ============================================================
# ML vs EGARCH_t_VIX: error and volatility-regime analysis
# Read-only analysis of saved initial predictions.
# ============================================================

import numpy as np
import pandas as pd

prediction_file = ML_OUTPUT_DIR / "initial_model_predictions.csv"

predictions = pd.read_csv(prediction_file, parse_dates=["Target_Date"])

required_columns = [
    "Target_Date",
    "Actual_Abs_Return",
    "EGARCH_t_VIX",
    "ML_Random_Forest",
]

assert all(c in predictions.columns for c in required_columns)
assert len(predictions) == 673
assert predictions["Target_Date"].is_monotonic_increasing
assert predictions["Target_Date"].is_unique
assert np.isfinite(
    predictions[required_columns[1:]].to_numpy(dtype=float)
).all()

# Absolute forecast errors
predictions["EGARCH_Abs_Error"] = (
    predictions["Actual_Abs_Return"] - predictions["EGARCH_t_VIX"]
).abs()

predictions["RF_Abs_Error"] = (
    predictions["Actual_Abs_Return"] - predictions["ML_Random_Forest"]
).abs()

predictions["RF_minus_EGARCH_Error"] = (
    predictions["RF_Abs_Error"] - predictions["EGARCH_Abs_Error"]
)

# Define regimes using actual volatility thresholds derived from
# the evaluation sample. These are descriptive labels, not trading rules.
q1 = predictions["Actual_Abs_Return"].quantile(1 / 3)
q2 = predictions["Actual_Abs_Return"].quantile(2 / 3)

predictions["Volatility_Regime"] = pd.cut(
    predictions["Actual_Abs_Return"],
    bins=[-np.inf, q1, q2, np.inf],
    labels=["Lower", "Middle", "Higher"],
    include_lowest=True,
)

regime_rows = []

for regime, group in predictions.groupby(
    "Volatility_Regime", observed=True
):
    for model_name, error_column in [
        ("EGARCH_t_VIX", "EGARCH_Abs_Error"),
        ("ML_Random_Forest", "RF_Abs_Error"),
    ]:
        regime_rows.append({
            "Regime": str(regime),
            "Model": model_name,
            "N": len(group),
            "MAE": group[error_column].mean(),
            "RMSE": np.sqrt(
                np.mean(
                    (
                        group["Actual_Abs_Return"]
                        - group[model_name]
                    ) ** 2
                )
            ),
        })

regime_results = pd.DataFrame(regime_rows)

regime_file = ML_OUTPUT_DIR / "rf_vs_egarch_vix_regime_analysis.csv"
regime_results.to_csv(regime_file, index=False)

# Overall paired comparison
overall_comparison = pd.DataFrame([{
    "Evaluation_Observations": len(predictions),
    "RF_Better_Absolute_Error_Count": int(
        (predictions["RF_minus_EGARCH_Error"] < 0).sum()
    ),
    "EGARCH_Better_Absolute_Error_Count": int(
        (predictions["RF_minus_EGARCH_Error"] > 0).sum()
    ),
    "Tied_Absolute_Error_Count": int(
        (predictions["RF_minus_EGARCH_Error"] == 0).sum()
    ),
    "Mean_RF_Minus_EGARCH_Absolute_Error":
        predictions["RF_minus_EGARCH_Error"].mean(),
}])

overall_file = ML_OUTPUT_DIR / "rf_vs_egarch_vix_error_comparison.csv"
overall_comparison.to_csv(overall_file, index=False)

print("Overall paired error comparison:")
display(overall_comparison.round(8))

print("\nPerformance by descriptive volatility regime:")
display(regime_results.round(8))

print("\nSaved:")
print(regime_file)
print(overall_file)

print("\nFrozen research outputs modified: NO")
print("Trading rules or thresholds modified: NO")


Overall paired error comparison:


,Evaluation_Observations,RF_Better_Absolute_Error_Count,EGARCH_Better_Absolute_Error_Count,Tied_Absolute_Error_Count,Mean_RF_Minus_EGARCH_Absolute_Error
0,673,276,397,0,0.000275



Performance by descriptive volatility regime:


,Regime,Model,N,MAE,RMSE
0,Lower,EGARCH_t_VIX,225,0.004163,0.004541
1,Lower,ML_Random_Forest,225,0.004549,0.004837
2,Middle,EGARCH_t_VIX,224,0.001645,0.002095
3,Middle,ML_Random_Forest,224,0.001751,0.002242
4,Higher,EGARCH_t_VIX,224,0.005403,0.007621
5,Higher,ML_Random_Forest,224,0.005736,0.008282



Saved:
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\rf_vs_egarch_vix_regime_analysis.csv
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\rf_vs_egarch_vix_error_comparison.csv

Frozen research outputs modified: NO
Trading rules or thresholds modified: NO


In [26]:

# ============================================================
# Monthly expanding-window walk-forward ML experiment
# Existing frozen statistical benchmarks remain unchanged.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use the saved initial predictions as the fixed comparison reference.
initial_predictions = pd.read_csv(
    ML_OUTPUT_DIR / "initial_model_predictions.csv",
    parse_dates=["Target_Date"],
)

benchmark_models = [
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
    "Historical_20D_VIX",
    "EWMA_VIX",
    "EGARCH_t_VIX",
]

assert len(initial_predictions) == 673
assert initial_predictions["Target_Date"].is_unique

# Evaluation schedule: forecast-origin date and target date.
schedule = evaluation_data[
    ["Date_Feature", "Target_Date"]
].copy()

schedule["Date_Feature"] = pd.to_datetime(schedule["Date_Feature"])
schedule["Target_Date"] = pd.to_datetime(schedule["Target_Date"])
schedule = schedule.sort_values("Target_Date").reset_index(drop=True)

assert len(schedule) == 673
assert (schedule["Date_Feature"] < schedule["Target_Date"]).all()

# Start with the same target dates and frozen benchmark forecasts.
walk_forward_predictions = initial_predictions[
    [
        "Target_Date",
        "Actual_Abs_Return",
        *benchmark_models,
        "ML_Random_Forest",
        "ML_Hist_Gradient_Boosting",
    ]
].copy()

walk_forward_predictions = walk_forward_predictions.merge(
    schedule,
    on="Target_Date",
    how="left",
    validate="one_to_one",
)

walk_forward_predictions["WF_ML_Random_Forest"] = np.nan
walk_forward_predictions["WF_ML_Hist_Gradient_Boosting"] = np.nan

# Monthly batches. Each batch is predicted by models fitted only
# on labels available by the first forecast-origin date of that batch.
walk_forward_predictions["Evaluation_Month"] = (
    walk_forward_predictions["Target_Date"].dt.to_period("M")
)

for month, batch in walk_forward_predictions.groupby(
    "Evaluation_Month", sort=True
):
    first_feature_date = batch["Date_Feature"].min()

    available_training = ml_dataset.loc[
        ml_dataset["Target_Date"] <= first_feature_date
    ].copy()

    assert not available_training.empty
    assert (
        available_training["Target_Date"].max()
        <= first_feature_date
    )

    X_train_wf = available_training[feature_columns]
    y_train_wf = available_training["Target_Abs_Return"]

    # Batch evaluation features are matched to the original evaluation rows.
    target_dates = batch["Target_Date"]
    eval_rows = evaluation_data.loc[
        evaluation_data["Target_Date"].isin(target_dates)
    ].sort_values("Target_Date")

    assert len(eval_rows) == len(batch)
    assert eval_rows["Target_Date"].reset_index(drop=True).equals(
        target_dates.reset_index(drop=True)
    )

    X_batch = eval_rows[feature_columns]

    rf = RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=10,
        max_features=0.8,
        random_state=SEED,
        n_jobs=-1,
    )

    hgb = HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=SEED,
    )

    rf.fit(X_train_wf, y_train_wf)
    hgb.fit(X_train_wf, y_train_wf)

    rf_pred = rf.predict(X_batch)
    hgb_pred = hgb.predict(X_batch)

    assert np.isfinite(rf_pred).all()
    assert np.isfinite(hgb_pred).all()

    # Assign predictions by target date, preserving row alignment.
    rf_map = dict(zip(eval_rows["Target_Date"], rf_pred))
    hgb_map = dict(zip(eval_rows["Target_Date"], hgb_pred))

    mask = walk_forward_predictions["Evaluation_Month"] == month

    walk_forward_predictions.loc[mask, "WF_ML_Random_Forest"] = (
        walk_forward_predictions.loc[mask, "Target_Date"].map(rf_map)
    )
    walk_forward_predictions.loc[
        mask, "WF_ML_Hist_Gradient_Boosting"
    ] = (
        walk_forward_predictions.loc[mask, "Target_Date"].map(hgb_map)
    )

    print(
        f"{month}: train={len(available_training)}, "
        f"evaluate={len(batch)}, "
        f"latest training target="
        f"{available_training['Target_Date'].max().date()}"
    )

assert walk_forward_predictions[
    ["WF_ML_Random_Forest", "WF_ML_Hist_Gradient_Boosting"]
].notna().all().all()

# Compare all models on identical target dates.
comparison_rows = []

comparison_columns = {
    **{name: name for name in benchmark_models},
    "ML_Random_Forest_Fixed": "ML_Random_Forest",
    "ML_Hist_Gradient_Boosting_Fixed": "ML_Hist_Gradient_Boosting",
    "ML_Random_Forest_Walk_Forward": "WF_ML_Random_Forest",
    "ML_Hist_Gradient_Boosting_Walk_Forward":
        "WF_ML_Hist_Gradient_Boosting",
}

for display_name, prediction_column in comparison_columns.items():
    actual = walk_forward_predictions["Actual_Abs_Return"].to_numpy()
    predicted = walk_forward_predictions[prediction_column].to_numpy()

    comparison_rows.append({
        "Model": display_name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "N": len(actual),
    })

walk_forward_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["RMSE", "MAE"])
    .reset_index(drop=True)
)

# Save only to the separate ML experiment directory.
predictions_path = (
    ML_OUTPUT_DIR / "walk_forward_predictions.csv"
)
comparison_path = (
    ML_OUTPUT_DIR / "walk_forward_model_comparison.csv"
)

walk_forward_predictions.to_csv(predictions_path, index=False)
walk_forward_comparison.to_csv(comparison_path, index=False)

print("\nWalk-forward model comparison (lower is better):")
display(walk_forward_comparison.round(8))

print("\nSaved:")
print(predictions_path)
print(comparison_path)

print("\nEvaluation observations:", len(walk_forward_predictions))
print("Frozen research outputs modified: NO")


2024-01: train=1456, evaluate=21, latest training target=2023-12-29
2024-02: train=1477, evaluate=21, latest training target=2024-01-31
2024-03: train=1498, evaluate=18, latest training target=2024-02-29
2024-04: train=1516, evaluate=20, latest training target=2024-03-28
2024-05: train=1536, evaluate=21, latest training target=2024-04-30
2024-06: train=1557, evaluate=19, latest training target=2024-05-31
2024-07: train=1576, evaluate=22, latest training target=2024-06-28
2024-08: train=1598, evaluate=21, latest training target=2024-07-31
2024-09: train=1619, evaluate=21, latest training target=2024-08-30
2024-10: train=1640, evaluate=22, latest training target=2024-09-30
2024-11: train=1662, evaluate=19, latest training target=2024-10-31
2024-12: train=1681, evaluate=21, latest training target=2024-11-29
2025-01: train=1702, evaluate=23, latest training target=2024-12-31
2025-02: train=1725, evaluate=20, latest training target=2025-01-31
2025-03: train=1745, evaluate=19, latest trainin

,Model,MAE,RMSE,N
0,EGARCH_t_VIX,0.003737,0.005262,673
1,Historical_20D_VIX,0.003863,0.005630,673
2,EWMA_VIX,0.003863,0.005637,673
3,ML_Random_Forest_Fixed,0.004013,0.005685,673
4,ML_Random_Forest_Walk_Forward,0.004012,0.005687,673
5,ML_Hist_Gradient_Boosting_Fixed,0.004052,0.005767,673
6,EGARCH_t,0.004628,0.005796,673
7,ML_Hist_Gradient_Boosting_Walk_Forward,0.004105,0.005920,673
8,GARCH_t,0.004852,0.006291,673
9,GJR_GARCH_t,0.004865,0.006349,673



Saved:
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\walk_forward_predictions.csv
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\walk_forward_model_comparison.csv

Evaluation observations: 673
Frozen research outputs modified: NO


In [27]:

# ============================================================
# Formal forecast comparison: ML vs EGARCH_t_VIX
# HAC-adjusted test of mean loss differential
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import norm
import statsmodels.api as sm

comparison_data = pd.read_csv(
    ML_OUTPUT_DIR / "walk_forward_predictions.csv",
    parse_dates=["Target_Date"],
).sort_values("Target_Date").reset_index(drop=True)

benchmark = "EGARCH_t_VIX"

ml_models = [
    "ML_Random_Forest",
    "WF_ML_Random_Forest",
    "WF_ML_Hist_Gradient_Boosting",
]

assert len(comparison_data) == 673
assert comparison_data["Target_Date"].is_unique

actual = comparison_data["Actual_Abs_Return"].to_numpy(dtype=float)
rows = []

for model_name in ml_models:
    prediction = comparison_data[model_name].to_numpy(dtype=float)
    benchmark_prediction = comparison_data[benchmark].to_numpy(dtype=float)

    assert np.isfinite(prediction).all()
    assert np.isfinite(benchmark_prediction).all()

    # Positive differential means the ML model has greater loss.
    loss_pairs = {
        "Absolute error": (
            np.abs(actual - prediction)
            - np.abs(actual - benchmark_prediction)
        ),
        "Squared error": (
            (actual - prediction) ** 2
            - (actual - benchmark_prediction) ** 2
        ),
    }

    for loss_name, differential in loss_pairs.items():
        # HAC lag of 5 trading observations; report this choice explicitly.
        fitted = sm.OLS(
            differential,
            np.ones((len(differential), 1)),
        ).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": 5},
            use_t=True,
        )

        mean_difference = float(np.mean(differential))
        standard_error = float(fitted.bse[0])
        statistic = (
            mean_difference / standard_error
            if standard_error > 0 else np.nan
        )
        p_value = float(fitted.pvalues[0])

        rows.append({
            "ML_Model": model_name,
            "Loss": loss_name,
            "Mean_Loss_Difference_ML_minus_EGARCH": mean_difference,
            "HAC_Standard_Error": standard_error,
            "Test_Statistic": statistic,
            "P_Value": p_value,
            "HAC_Max_Lag": 5,
            "N": len(differential),
        })

formal_comparison = pd.DataFrame(rows)

formal_path = (
    ML_OUTPUT_DIR / "formal_ml_vs_egarch_comparison.csv"
)
formal_comparison.to_csv(formal_path, index=False)

print("HAC-adjusted forecast-loss comparison")
print("Positive mean difference = ML has higher loss.")
print("Negative mean difference = ML has lower loss.")
display(formal_comparison.round(8))

print("\nSaved:", formal_path)
print("Frozen research outputs modified: NO")


HAC-adjusted forecast-loss comparison
Positive mean difference = ML has higher loss.
Negative mean difference = ML has lower loss.


,ML_Model,Loss,Mean_Loss_Difference_ML_minus_EGARCH,HAC_Standard_Error,Test_Statistic,P_Value,HAC_Max_Lag,N
0,ML_Random_Forest,Absolute error,0.000275,5.238000e-05,5.252776,2.000000e-07,5,673
1,ML_Random_Forest,Squared error,0.000005,9.900000e-07,4.674400,3.560000e-06,5,673
2,WF_ML_Random_Forest,Absolute error,0.000275,5.005000e-05,5.491296,6.000000e-08,5,673
3,WF_ML_Random_Forest,Squared error,0.000005,1.030000e-06,4.498367,8.070000e-06,5,673
4,WF_ML_Hist_Gradient_Boosting,Absolute error,0.000367,8.344000e-05,4.400741,1.254000e-05,5,673
5,WF_ML_Hist_Gradient_Boosting,Squared error,0.000007,2.030000e-06,3.633742,3.006500e-04,5,673



Saved: D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\formal_ml_vs_egarch_comparison.csv
Frozen research outputs modified: NO


In [28]:

# ============================================================
# CELL 15 — Mix Strategy construction and evaluation
# Source: saved, date-aligned forecast predictions
# Frozen research outputs remain read-only.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

SOURCE_FILE = ML_OUTPUT_DIR / "walk_forward_predictions.csv"

mix_data = pd.read_csv(
    SOURCE_FILE,
    parse_dates=["Target_Date"],
).sort_values("Target_Date").reset_index(drop=True)

# Five NIFTY-only statistical forecasts.
nifty_models = [
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
]

# Three forecasts enhanced with India VIX.
vix_models = [
    "Historical_20D_VIX",
    "EWMA_VIX",
    "EGARCH_t_VIX",
]

all_statistical_models = nifty_models + vix_models

required_columns = [
    "Target_Date",
    "Actual_Abs_Return",
    *all_statistical_models,
]

assert all(c in mix_data.columns for c in required_columns)
assert len(mix_data) == 673
assert mix_data["Target_Date"].is_unique
assert mix_data["Target_Date"].is_monotonic_increasing
assert np.isfinite(
    mix_data[required_columns[1:]].to_numpy(dtype=float)
).all()

# ------------------------------------------------------------
# Strategy 1: Equal-weight average of five NIFTY-only forecasts
# ------------------------------------------------------------
mix_data["Mix_Strategy_NIFTY_Only"] = (
    mix_data[nifty_models].mean(axis=1)
)

# ------------------------------------------------------------
# Strategy 2: Equal-weight average of all eight forecasts
# ------------------------------------------------------------
mix_data["Mix_Strategy_All_Forecasts"] = (
    mix_data[all_statistical_models].mean(axis=1)
)

# ------------------------------------------------------------
# Strategy 3: Equal weighting of the two forecast groups
# Each group receives 50%, regardless of group size.
# ------------------------------------------------------------
nifty_group_mean = mix_data[nifty_models].mean(axis=1)
vix_group_mean = mix_data[vix_models].mean(axis=1)

mix_data["Mix_Strategy_Group_Balanced"] = (
    0.50 * nifty_group_mean
    + 0.50 * vix_group_mean
)

# ------------------------------------------------------------
# Evaluate all candidates on identical target dates
# ------------------------------------------------------------
comparison_columns = {
    **{name: name for name in all_statistical_models},
    "Mix_Strategy_NIFTY_Only": "Mix_Strategy_NIFTY_Only",
    "Mix_Strategy_All_Forecasts": "Mix_Strategy_All_Forecasts",
    "Mix_Strategy_Group_Balanced": "Mix_Strategy_Group_Balanced",
}

# Include ML results for context if present in the saved file.
for name in [
    "ML_Random_Forest",
    "WF_ML_Random_Forest",
    "ML_Hist_Gradient_Boosting",
    "WF_ML_Hist_Gradient_Boosting",
]:
    if name in mix_data.columns:
        comparison_columns[name] = name

actual = mix_data["Actual_Abs_Return"].to_numpy(dtype=float)
result_rows = []

for model_name, prediction_column in comparison_columns.items():
    predicted = mix_data[prediction_column].to_numpy(dtype=float)

    assert len(predicted) == len(actual)
    assert np.isfinite(predicted).all()

    result_rows.append({
        "Model": model_name,
        "Type": (
            "Mix Strategy"
            if model_name.startswith("Mix_Strategy_")
            else (
                "Machine learning"
                if model_name.startswith(("ML_", "WF_ML_"))
                else "Statistical benchmark"
            )
        ),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "N": len(actual),
    })

mix_comparison = (
    pd.DataFrame(result_rows)
    .sort_values(["RMSE", "MAE"])
    .reset_index(drop=True)
)

# Save new experiment outputs only in the ML experiment directory.
mix_predictions_path = (
    ML_OUTPUT_DIR / "mix_strategy_predictions.csv"
)
mix_comparison_path = (
    ML_OUTPUT_DIR / "mix_strategy_comparison.csv"
)

mix_data.to_csv(mix_predictions_path, index=False)
mix_comparison.to_csv(mix_comparison_path, index=False)

print("Mix Strategy comparison — lower MAE/RMSE is better:")
display(mix_comparison.round(8))

print("\nMix forecast correlations:")
display(
    mix_data[
        all_statistical_models
        + [
            "Mix_Strategy_NIFTY_Only",
            "Mix_Strategy_All_Forecasts",
            "Mix_Strategy_Group_Balanced",
        ]
    ].corr().round(4)
)

print("\nSaved:")
print(mix_predictions_path)
print(mix_comparison_path)

print("\nEvaluation observations:", len(mix_data))
print("Frozen research outputs modified: NO")


Mix Strategy comparison — lower MAE/RMSE is better:


,Model,Type,MAE,RMSE,N
0,EGARCH_t_VIX,Statistical benchmark,0.003737,0.005262,673
1,Mix_Strategy_Group_Balanced,Mix Strategy,0.004119,0.005584,673
2,Historical_20D_VIX,Statistical benchmark,0.003863,0.005630,673
3,EWMA_VIX,Statistical benchmark,0.003863,0.005637,673
4,Mix_Strategy_All_Forecasts,Mix Strategy,0.004252,0.005684,673
5,ML_Random_Forest,Machine learning,0.004013,0.005685,673
6,WF_ML_Random_Forest,Machine learning,0.004012,0.005687,673
7,ML_Hist_Gradient_Boosting,Machine learning,0.004052,0.005767,673
8,EGARCH_t,Statistical benchmark,0.004628,0.005796,673
9,WF_ML_Hist_Gradient_Boosting,Machine learning,0.004105,0.005920,673



Mix forecast correlations:


,Historical_20D,EWMA,GARCH_t,GJR_GARCH_t,EGARCH_t,Historical_20D_VIX,EWMA_VIX,EGARCH_t_VIX,Mix_Strategy_NIFTY_Only,Mix_Strategy_All_Forecasts,Mix_Strategy_Group_Balanced
Historical_20D,1.0000,0.8954,0.7070,0.5672,0.6531,0.5239,0.4554,0.6747,0.8607,0.8185,0.7899
EWMA,0.8954,1.0000,0.7747,0.6308,0.6838,0.5997,0.5027,0.7239,0.8927,0.8614,0.8366
GARCH_t,0.7070,0.7747,1.0000,0.8942,0.8339,0.5859,0.5203,0.8081,0.9416,0.9080,0.8817
GJR_GARCH_t,0.5672,0.6308,0.8942,1.0000,0.8591,0.5854,0.5391,0.8196,0.8828,0.8686,0.8507
EGARCH_t,0.6531,0.6838,0.8339,0.8591,1.0000,0.6797,0.6364,0.9509,0.8932,0.9135,0.9089
Historical_20D_VIX,0.5239,0.5997,0.5859,0.5854,0.6797,1.0000,0.9922,0.8714,0.6602,0.8069,0.8552
EWMA_VIX,0.4554,0.5027,0.5203,0.5391,0.6364,0.9922,1.0000,0.8375,0.5884,0.7503,0.8055
EGARCH_t_VIX,0.6747,0.7239,0.8081,0.8196,0.9509,0.8714,0.8375,1.0000,0.8824,0.9544,0.9703
Mix_Strategy_NIFTY_Only,0.8607,0.8927,0.9416,0.8828,0.8932,0.6602,0.5884,0.8824,1.0000,0.9754,0.9518
Mix_Strategy_All_Forecasts,0.8185,0.8614,0.9080,0.8686,0.9135,0.8069,0.7503,0.9544,0.9754,1.0000,0.9960



Saved:
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\mix_strategy_predictions.csv
D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\mix_strategy_comparison.csv

Evaluation observations: 673
Frozen research outputs modified: NO


In [29]:
# ============================================================
# CELL 16 — Incremental VIX Value and Mix Strategy Comparison
# Uses the existing evaluation period; exploratory inference.
# Frozen research outputs remain read-only.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

SOURCE_FILE = ML_OUTPUT_DIR / "mix_strategy_predictions.csv"

vix_test = pd.read_csv(SOURCE_FILE, parse_dates=["Target_Date"])
vix_test = vix_test.sort_values("Target_Date").reset_index(drop=True)

actual = vix_test["Actual_Abs_Return"].to_numpy(dtype=float)

paired_comparisons = [
    ("Historical_20D", "Historical_20D_VIX"),
    ("EWMA", "EWMA_VIX"),
    ("EGARCH_t", "EGARCH_t_VIX"),
    ("EGARCH_t_VIX", "Mix_Strategy_Group_Balanced"),
]

rows = []

for baseline, challenger in paired_comparisons:
    y_base = vix_test[baseline].to_numpy(dtype=float)
    y_new = vix_test[challenger].to_numpy(dtype=float)

    base_abs_loss = np.abs(actual - y_base)
    new_abs_loss = np.abs(actual - y_new)

    base_sq_loss = (actual - y_base) ** 2
    new_sq_loss = (actual - y_new) ** 2

    # Positive difference means the challenger has lower loss.
    abs_gain = base_abs_loss - new_abs_loss
    sq_gain = base_sq_loss - new_sq_loss

    # HAC inference on paired loss differentials.
    abs_fit = sm.OLS(abs_gain, np.ones((len(abs_gain), 1))).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5},
        use_t=True,
    )
    sq_fit = sm.OLS(sq_gain, np.ones((len(sq_gain), 1))).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5},
        use_t=True,
    )

    rows.append({
        "Baseline": baseline,
        "Challenger": challenger,
        "Baseline_MAE": mean_absolute_error(actual, y_base),
        "Challenger_MAE": mean_absolute_error(actual, y_new),
        "MAE_Improvement": (
            mean_absolute_error(actual, y_base)
            - mean_absolute_error(actual, y_new)
        ),
        "Baseline_RMSE": np.sqrt(mean_squared_error(actual, y_base)),
        "Challenger_RMSE": np.sqrt(mean_squared_error(actual, y_new)),
        "RMSE_Improvement": (
            np.sqrt(mean_squared_error(actual, y_base))
            - np.sqrt(mean_squared_error(actual, y_new))
        ),
        "Mean_Absolute_Loss_Gain": abs_gain.mean(),
        "HAC_t_Absolute_Loss": float(abs_fit.tvalues[0]),
        "HAC_p_Absolute_Loss": float(abs_fit.pvalues[0]),
        "Mean_Squared_Loss_Gain": sq_gain.mean(),
        "HAC_t_Squared_Loss": float(sq_fit.tvalues[0]),
        "HAC_p_Squared_Loss": float(sq_fit.pvalues[0]),
        "N": len(actual),
    })

vix_incremental_results = pd.DataFrame(rows)

OUTPUT_FILE = ML_OUTPUT_DIR / "vix_incremental_value_analysis.csv"
vix_incremental_results.to_csv(OUTPUT_FILE, index=False)

display(vix_incremental_results.round(8))
print("\nSaved:", OUTPUT_FILE)
print("Frozen research outputs modified: NO")

,Baseline,Challenger,Baseline_MAE,Challenger_MAE,MAE_Improvement,Baseline_RMSE,Challenger_RMSE,RMSE_Improvement,Mean_Absolute_Loss_Gain,HAC_t_Absolute_Loss,HAC_p_Absolute_Loss,Mean_Squared_Loss_Gain,HAC_t_Squared_Loss,HAC_p_Squared_Loss,N
0,Historical_20D,Historical_20D_VIX,0.004923,0.003863,0.001059,0.006564,0.005630,0.000934,0.001059,4.972570,8.400000e-07,0.000011,3.205722,0.001411,673
1,EWMA,EWMA_VIX,0.004924,0.003863,0.001061,0.006419,0.005637,0.000782,0.001061,5.944226,0.000000e+00,0.000009,3.704630,0.000229,673
2,EGARCH_t,EGARCH_t_VIX,0.004628,0.003737,0.000890,0.005796,0.005262,0.000535,0.000890,9.598817,0.000000e+00,0.000006,4.104787,0.000045,673
3,EGARCH_t_VIX,Mix_Strategy_Group_Balanced,0.003737,0.004119,-0.000381,0.005262,0.005584,-0.000322,-0.000381,-9.616284,0.000000e+00,-0.000003,-6.244837,0.000000,673



Saved: D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\vix_incremental_value_analysis.csv
Frozen research outputs modified: NO


### Incremental India VIX Value and Forecast-Mix Evaluation

**Objective:** Evaluate whether India VIX improves matched volatility forecasts and whether combining statistical forecasts outperforms the strongest individual benchmark.

**Key findings**
- VIX-enhanced versions of Historical Volatility, EWMA, and EGARCH-t achieved lower MAE and RMSE than their corresponding NIFTY-only models over the 673-observation evaluation period.
- `EGARCH_t_VIX` remained the best-performing model, with MAE of 0.003737 and RMSE of 0.005262.
- The group-balanced mix underperformed `EGARCH_t_VIX`, with MAE of 0.004119 and RMSE of 0.005584.
- HAC-adjusted paired loss comparisons favored the VIX-enhanced matched models and the standalone `EGARCH_t_VIX` benchmark over the group-balanced mix under the selected specification.

**Research interpretation:** India VIX appears to provide useful incremental forecasting information in this evaluation sample. However, the equal-weight mixing strategies did not improve forecast accuracy over the best individual model. These findings are exploratory and do not establish performance in future market conditions.

**Decision and next step:** Retain `EGARCH_t_VIX` as the leading benchmark. Before developing an ML stacking strategy, identify or construct historical out-of-sample base-model forecasts for the pre-2024 training period. The meta-model must be trained without in-sample base forecasts or future information.

**Output:** `data/ml_experiments/v1/vix_incremental_value_analysis.csv`

**Integrity:** Frozen research outputs remain unchanged.

In [30]:
# ============================================================
# Locate Existing Historical Forecast Artifacts
# Read-only filesystem inspection
# ============================================================

from pathlib import Path
import pandas as pd

SEARCH_DIRS = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "Notebooks",
]

KEYWORDS = (
    "forecast",
    "validation",
    "walk_forward",
    "oos",
    "prediction",
    "calibration",
)

DATE_CANDIDATES = ("Date", "Target_Date", "Forecast_Date")
found_files = []

for search_dir in SEARCH_DIRS:
    if not search_dir.exists():
        continue

    for path in search_dir.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in {".csv", ".parquet"}:
            continue

        if not any(word in path.stem.lower() for word in KEYWORDS):
            continue

        found_files.append(path)

print(f"Candidate files found: {len(found_files)}\n")

inventory = []

for path in sorted(set(found_files)):
    try:
        if path.suffix.lower() == ".csv":
            sample = pd.read_csv(path, nrows=5)
            row_count = sum(
                1 for _ in path.open("r", encoding="utf-8-sig")
            ) - 1
        else:
            sample = pd.read_parquet(path)
            row_count = len(sample)

        columns = list(sample.columns)
        date_col = next(
            (c for c in DATE_CANDIDATES if c in columns),
            None,
        )

        date_range = "Not identified"

        if date_col is not None and len(sample):
            if path.suffix.lower() == ".csv":
                dates = pd.read_csv(path, usecols=[date_col])[date_col]
            else:
                dates = pd.read_parquet(path, columns=[date_col])[date_col]

            dates = pd.to_datetime(dates, errors="coerce").dropna()

            if not dates.empty:
                date_range = (
                    f"{dates.min().date()} to {dates.max().date()}"
                )

        inventory.append({
            "File": str(path.relative_to(PROJECT_ROOT)),
            "Rows": row_count,
            "Date_Column": date_col,
            "Date_Range": date_range,
            "Forecast_Columns": [
                c for c in columns
                if any(k in c.lower() for k in
                       ("garch", "egarch", "ewma", "historical",
                        "forecast", "vix", "actual", "return"))
            ],
        })

    except Exception as exc:
        inventory.append({
            "File": str(path.relative_to(PROJECT_ROOT)),
            "Rows": "Read error",
            "Date_Column": None,
            "Date_Range": str(exc),
            "Forecast_Columns": [],
        })

if inventory:
    display(pd.DataFrame(inventory).to_string(index=False))
else:
    print("No matching CSV or Parquet artifacts found.")

print("\nRead-only inspection complete. No files changed.")

Candidate files found: 7



'                                                           File  Rows Date_Column               Date_Range                                                                                                                                             Forecast_Columns\n                      data\\business_inputs\\v1\\oos_forecasts.csv   673        Date 2024-01-01 to 2026-09-22 [NIFTY_Log_Return, Actual_Volatility, India_VIX_PreReturn, Historical_20D, EWMA, GARCH_t, GJR_GARCH_t, EGARCH_t, Historical_20D_VIX, EWMA_VIX, EGARCH_t_VIX]\ndata\\ml_experiments\\v1\\historical_validation_base_forecasts.csv   493        Date 2022-01-03 to 2023-12-29                                                                                    [Actual_Volatility, Historical_20D, EWMA, GARCH_t, GJR_GARCH_t, EGARCH_t]\n           data\\ml_experiments\\v1\\initial_model_predictions.csv   673 Target_Date 2024-01-01 to 2026-09-22                                        [Actual_Abs_Return, Historical_20D, EWMA, GARCH_t


Read-only inspection complete. No files changed.


In [31]:
# ============================================================
# CELL 18 — Locate Historical Forecast Logic in Existing Notebook
# Read-only inspection; does not execute or modify the notebook
# ============================================================

import json
from pathlib import Path

NOTEBOOK_FILE = (
    PROJECT_ROOT / "Notebooks" /
    "02_market_risk_research_organized.ipynb"
)

assert NOTEBOOK_FILE.is_file(), (
    f"Notebook not found: {NOTEBOOK_FILE}"
)

with NOTEBOOK_FILE.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = (
    "validation_forecasts",
    "walk_forward",
    "out_of_sample",
    "out-of-sample",
    "expanding",
    "forecast",
)

matches = []

for cell_index, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))

    if any(word in source.lower() for word in keywords):
        matching_lines = [
            (line_number, line)
            for line_number, line in enumerate(
                source.splitlines(), start=1
            )
            if any(word in line.lower() for word in keywords)
        ]

        if matching_lines:
            matches.append({
                "cell": cell_index,
                "type": cell.get("cell_type"),
                "lines": matching_lines,
                "source": source,
            })

print("Notebook:", NOTEBOOK_FILE)
print("Total cells:", len(notebook.get("cells", [])))
print("Matching cells:", len(matches))

for item in matches:
    print(f"\n{'=' * 72}")
    print(f"CELL INDEX: {item['cell']} | TYPE: {item['type']}")
    for line_number, line in item["lines"][:15]:
        print(f"{line_number:>4}: {line[:220]}")

print("\nRead-only inspection complete. Notebook was not executed or modified.")

Notebook: D:\DS Projects\Adaptive Market Risk Intelligence System\Notebooks\02_market_risk_research_organized.ipynb
Total cells: 552
Matching cells: 292

CELL INDEX: 0 | TYPE: markdown
  14: Data acquisition → validation → return construction → volatility baselines → statistical diagnostics → asymmetric models → OOS forecasting → validation selection → forecast combinations → incremental market information →

CELL INDEX: 2 | TYPE: markdown
  10: | Research table | `tbl_analysis_metric.csv` | `tbl_oos_volatility_forecast_performance.csv` |
  12: | Research log | `research_log_stage.md` | `research_log_forecast_combination.md` |

CELL INDEX: 15 | TYPE: markdown
  13: out-of-sample evaluation periods.

CELL INDEX: 22 | TYPE: markdown
  11: final forecasting dataset.

CELL INDEX: 23 | TYPE: markdown
  20: one-step-ahead forecasting.
  23: out-of-sample forecasting framework.

CELL INDEX: 24 | TYPE: markdown
  10: forecasts using the predefined chronological validation and out-of-sample

CE

In [35]:

# ============================================================
# HISTORICAL VALIDATION INPUT — READ-ONLY LOAD AND VALIDATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

HISTORICAL_VALIDATION_FILE = (
    PROJECT_ROOT
    / "data"
    / "ml_experiments"
    / "v1"
    / "historical_validation_base_forecasts.csv"
)

assert HISTORICAL_VALIDATION_FILE.is_file(), (
    f"Historical validation file not found:\n"
    f"{HISTORICAL_VALIDATION_FILE}"
)

historical_validation = pd.read_csv(
    HISTORICAL_VALIDATION_FILE,
    parse_dates=["Date"],
)

required_columns = [
    "Date",
    "Actual_Volatility",
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
]

assert historical_validation.columns.tolist() == required_columns
assert len(historical_validation) == 493
assert historical_validation["Date"].is_unique
assert historical_validation["Date"].is_monotonic_increasing
assert historical_validation["Date"].min() == pd.Timestamp("2022-01-03")
assert historical_validation["Date"].max() == pd.Timestamp("2023-12-29")
assert historical_validation[required_columns[1:]].notna().all().all()
assert np.isfinite(
    historical_validation[required_columns[1:]].to_numpy(dtype=float)
).all()

print("Historical validation input: PASS")
print("Shape:", historical_validation.shape)
print(
    "Date range:",
    historical_validation["Date"].min().date(),
    "to",
    historical_validation["Date"].max().date(),
)
print("Source: existing CSV")
print("Forecasts regenerated: NO")
print("Frozen research outputs modified: NO")


Historical validation input: PASS
Shape: (493, 7)
Date range: 2022-01-03 to 2023-12-29
Source: existing CSV
Forecasts regenerated: NO
Frozen research outputs modified: NO


In [36]:

# ============================================================
# CELL 20 — Load and Validate Historical Stacking Data
# Read-only: does not regenerate forecasts or overwrite files.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Locate the existing historical validation artifact
# ------------------------------------------------------------

PROJECT_ROOT_STACKING = Path(
    r"D:\DS Projects\Adaptive Market Risk Intelligence System"
)

HISTORICAL_VALIDATION_FILE = (
    PROJECT_ROOT_STACKING
    / "data"
    / "ml_experiments"
    / "v1"
    / "historical_validation_base_forecasts.csv"
)

assert HISTORICAL_VALIDATION_FILE.is_file(), (
    "Historical validation CSV not found:\n"
    f"{HISTORICAL_VALIDATION_FILE}\n"
    "Do not regenerate it automatically. Check the file location first."
)

# ------------------------------------------------------------
# 2. Load existing data into a dedicated variable
# ------------------------------------------------------------

historical_validation = pd.read_csv(
    HISTORICAL_VALIDATION_FILE,
    parse_dates=["Date"],
)

# ------------------------------------------------------------
# 3. Validate schema and chronology
# ------------------------------------------------------------

required_columns = [
    "Date",
    "Actual_Volatility",
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
]

assert historical_validation.columns.tolist() == required_columns, (
    "Unexpected historical validation schema.\n"
    f"Expected: {required_columns}\n"
    f"Found: {historical_validation.columns.tolist()}"
)

assert len(historical_validation) == 493, (
    f"Expected 493 observations; found {len(historical_validation)}"
)

assert historical_validation["Date"].notna().all()
assert historical_validation["Date"].is_unique
assert historical_validation["Date"].is_monotonic_increasing

assert (
    historical_validation["Date"].min()
    == pd.Timestamp("2022-01-03")
), "Unexpected validation start date."

assert (
    historical_validation["Date"].max()
    == pd.Timestamp("2023-12-29")
), "Unexpected validation end date."

numeric_columns = required_columns[1:]

assert historical_validation[numeric_columns].notna().all().all(), (
    "Missing values found in validation targets or forecasts."
)

assert np.isfinite(
    historical_validation[numeric_columns].to_numpy(dtype=float)
).all(), "Non-finite values found in validation data."

# Forecasts should be non-negative volatility estimates.
assert (
    historical_validation[numeric_columns] >= 0
).all().all(), "Negative volatility values detected."

# ------------------------------------------------------------
# 4. Report validation results
# ------------------------------------------------------------

print("=" * 68)
print("HISTORICAL VALIDATION DATA — INTEGRITY CHECK")
print("=" * 68)

print("Status: PASS")
print("Shape:", historical_validation.shape)
print(
    "Date range:",
    historical_validation["Date"].min().date(),
    "to",
    historical_validation["Date"].max().date(),
)
print("Columns:", historical_validation.columns.tolist())
print("Missing values:", int(
    historical_validation[required_columns].isna().sum().sum()
))
print("Source:", HISTORICAL_VALIDATION_FILE)
print("Operation: Read-only")
print("Frozen research outputs modified: NO")

display(historical_validation.head())


HISTORICAL VALIDATION DATA — INTEGRITY CHECK
Status: PASS
Shape: (493, 7)
Date range: 2022-01-03 to 2023-12-29
Columns: ['Date', 'Actual_Volatility', 'Historical_20D', 'EWMA', 'GARCH_t', 'GJR_GARCH_t', 'EGARCH_t']
Missing values: 0
Source: D:\DS Projects\Adaptive Market Risk Intelligence System\data\ml_experiments\v1\historical_validation_base_forecasts.csv
Operation: Read-only
Frozen research outputs modified: NO


,Date,Actual_Volatility,Historical_20D,EWMA,GARCH_t,GJR_GARCH_t,EGARCH_t
0,2022-01-03,0.015532,0.010488,0.012703,0.009016,0.009443,0.009120
1,2022-01-04,0.010135,0.010187,0.012890,0.009811,0.009035,0.008711
2,2022-01-05,0.006717,0.009880,0.012742,0.009765,0.008660,0.008285
3,2022-01-06,0.010056,0.009305,0.012463,0.009442,0.008317,0.009428
4,2022-01-07,0.003757,0.009634,0.012332,0.009682,0.009159,0.008891


In [39]:

# ============================================================
# CELL 21 — PREPARE STACKING TRAINING AND OOS DATA
# Read-only: no model fitting or file writes.
# ============================================================

import numpy as np
import pandas as pd

OOS_FORECAST_FILE = (
    PROJECT_ROOT
    / "data"
    / "research_outputs"
    / "v1"
    / "oos_forecasts.csv"
)

assert OOS_FORECAST_FILE.is_file(), (
    f"Frozen OOS forecast file not found:\n{OOS_FORECAST_FILE}"
)

stack_oos_source = pd.read_csv(
    OOS_FORECAST_FILE,
    parse_dates=["Date"],
)

stack_features = [
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
]

required_stack_columns = [
    "Date",
    "Actual_Volatility",
    *stack_features,
]

assert set(required_stack_columns).issubset(
    historical_validation.columns
), "Historical validation data is missing required columns."

assert set(required_stack_columns).issubset(
    stack_oos_source.columns
), (
    "Frozen OOS file is missing required columns.\n"
    f"Available columns: {stack_oos_source.columns.tolist()}"
)

# Historical training period: 2022–2023.
stack_train = (
    historical_validation[required_stack_columns]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

# Frozen final evaluation period: 2024–2026.
stack_test = (
    stack_oos_source[required_stack_columns]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

# Validate each dataset.
for label, frame in [
    ("Training", stack_train),
    ("Final OOS evaluation", stack_test),
]:
    assert frame["Date"].notna().all(), f"{label}: missing dates."
    assert frame["Date"].is_unique, f"{label}: duplicate dates."
    assert frame["Date"].is_monotonic_increasing
    assert frame[required_stack_columns[1:]].notna().all().all()
    assert np.isfinite(
        frame[required_stack_columns[1:]].to_numpy(dtype=float)
    ).all()
    assert (
        frame[required_stack_columns[1:]] >= 0
    ).all().all(), f"{label}: negative volatility values."

assert stack_train["Date"].max() < stack_test["Date"].min(), (
    "Training and final OOS evaluation periods overlap."
)

assert len(stack_train) == 493
assert len(stack_test) == 673

print("=" * 68)
print("STACKING DATA PREPARATION — PASS")
print("=" * 68)

print("Features:", stack_features)
print(
    "Training rows:", len(stack_train),
    "| Dates:",
    stack_train["Date"].min().date(),
    "to",
    stack_train["Date"].max().date(),
)
print(
    "Final OOS rows:", len(stack_test),
    "| Dates:",
    stack_test["Date"].min().date(),
    "to",
    stack_test["Date"].max().date(),
)
print("Training/OOS overlap: NO")
print("Frozen research outputs modified: NO")

display(stack_train.head(3))
display(stack_test.head(3))


STACKING DATA PREPARATION — PASS
Features: ['Historical_20D', 'EWMA', 'GARCH_t', 'GJR_GARCH_t', 'EGARCH_t']
Training rows: 493 | Dates: 2022-01-03 to 2023-12-29
Final OOS rows: 673 | Dates: 2024-01-01 to 2026-09-22
Training/OOS overlap: NO
Frozen research outputs modified: NO


,Date,Actual_Volatility,Historical_20D,EWMA,GARCH_t,GJR_GARCH_t,EGARCH_t
0,2022-01-03,0.015532,0.010488,0.012703,0.009016,0.009443,0.009120
1,2022-01-04,0.010135,0.010187,0.012890,0.009811,0.009035,0.008711
2,2022-01-05,0.006717,0.009880,0.012742,0.009765,0.008660,0.008285


,Date,Actual_Volatility,Historical_20D,EWMA,GARCH_t,GJR_GARCH_t,EGARCH_t
0,2024-01-01,0.000483,0.007280,0.011568,0.007547,0.006755,0.005899
1,2024-01-02,0.003506,0.007284,0.011216,0.006990,0.006427,0.006179
2,2024-01-03,0.006875,0.006246,0.010908,0.006801,0.006577,0.006909


In [40]:

# ============================================================
# CELL 22 — RIDGE STACKING MODEL AND OOS COMPARISON
# Training: 2022–2023
# Final OOS evaluation: 2024–2026
# No files written; frozen outputs remain untouched.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 1. Validate prepared datasets
# ------------------------------------------------------------

stack_features = [
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
]

assert len(stack_train) == 493
assert len(stack_test) == 673

assert stack_train["Date"].max() < stack_test["Date"].min()
assert stack_train[stack_features].notna().all().all()
assert stack_test[stack_features].notna().all().all()

X_stack_train = stack_train[stack_features].to_numpy(dtype=float)
y_stack_train = stack_train["Actual_Volatility"].to_numpy(dtype=float)

X_stack_test = stack_test[stack_features].to_numpy(dtype=float)
y_stack_test = stack_test["Actual_Volatility"].to_numpy(dtype=float)

assert np.isfinite(X_stack_train).all()
assert np.isfinite(y_stack_train).all()
assert np.isfinite(X_stack_test).all()
assert np.isfinite(y_stack_test).all()

# ------------------------------------------------------------
# 2. Tune Ridge regularization on historical training data only
# ------------------------------------------------------------

ridge_pipeline = make_pipeline(
    StandardScaler(),
    Ridge()
)

time_cv = TimeSeriesSplit(n_splits=5)

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={
        "ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
    },
    scoring="neg_mean_absolute_error",
    cv=time_cv,
    refit=True,
    n_jobs=None,
)

ridge_search.fit(X_stack_train, y_stack_train)

# GridSearchCV refits the selected model on all 493 training rows.
stacking_ridge_model = ridge_search.best_estimator_

# ------------------------------------------------------------
# 3. Generate final OOS predictions
# ------------------------------------------------------------

ridge_prediction_raw = stacking_ridge_model.predict(X_stack_test)

# Volatility forecasts cannot be negative.
ridge_prediction = np.maximum(ridge_prediction_raw, 0.0)

# ------------------------------------------------------------
# 4. Build a common comparison dataset
# ------------------------------------------------------------

stacking_predictions = pd.DataFrame({
    "Date": stack_test["Date"].to_numpy(),
    "Actual_Volatility": y_stack_test,
})

for feature in stack_features:
    stacking_predictions[feature] = (
        stack_test[feature].to_numpy()
    )

stacking_predictions["Equal_Weight_Mean"] = (
    stack_test[stack_features].mean(axis=1).to_numpy()
)

stacking_predictions["Equal_Weight_Median"] = (
    stack_test[stack_features].median(axis=1).to_numpy()
)

stacking_predictions["Ridge_Stacking"] = ridge_prediction

# ------------------------------------------------------------
# 5. Evaluate every forecast on the same OOS dates
# ------------------------------------------------------------

prediction_columns = stack_features + [
    "Equal_Weight_Mean",
    "Equal_Weight_Median",
    "Ridge_Stacking",
]

stacking_results = []

for model_name in prediction_columns:
    actual = stacking_predictions["Actual_Volatility"].to_numpy()
    predicted = stacking_predictions[model_name].to_numpy()

    stacking_results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "Mean_Forecast_Error": float(
            np.mean(predicted - actual)
        ),
    })

stacking_comparison = (
    pd.DataFrame(stacking_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Report results
# ------------------------------------------------------------

print("=" * 72)
print("RIDGE STACKING — FINAL OOS EVALUATION")
print("=" * 72)

print("Training rows:", len(stack_train))
print("Final OOS rows:", len(stack_test))
print(
    "OOS dates:",
    stack_test["Date"].min().date(),
    "to",
    stack_test["Date"].max().date(),
)
print("Selected Ridge alpha:",
      ridge_search.best_params_["ridge__alpha"])
print(
    "Best chronological CV MAE:",
    f"{-ridge_search.best_score_:.8f}",
)

print("\nOOS comparison — lower MAE and RMSE are better:")
display(stacking_comparison.round(8))

print("\nStandardized-model Ridge coefficients:")
ridge_fitted = stacking_ridge_model.named_steps["ridge"]

display(pd.DataFrame({
    "Feature": stack_features,
    "Coefficient": ridge_fitted.coef_,
}).round(8))

print("\nFrozen research outputs modified: NO")
print("New files written: NO")


RIDGE STACKING — FINAL OOS EVALUATION
Training rows: 493
Final OOS rows: 673
OOS dates: 2024-01-01 to 2026-09-22
Selected Ridge alpha: 10.0
Best chronological CV MAE: 0.00357537

OOS comparison — lower MAE and RMSE are better:


,Model,MAE,RMSE,Mean_Forecast_Error
0,Ridge_Stacking,0.003700,0.005182,0.000426
1,EGARCH_t,0.004628,0.005796,0.002651
2,Equal_Weight_Median,0.004733,0.006136,0.002219
3,Equal_Weight_Mean,0.004767,0.006140,0.002303
4,GARCH_t,0.004852,0.006291,0.002361
5,GJR_GARCH_t,0.004865,0.006349,0.002447
6,Historical_20D,0.004923,0.006564,0.001883
7,EWMA,0.004924,0.006419,0.002174



Standardized-model Ridge coefficients:


,Feature,Coefficient
0,Historical_20D,-0.001331
1,EWMA,0.001870
2,GARCH_t,-0.001568
3,GJR_GARCH_t,-0.002691
4,EGARCH_t,0.006307



Frozen research outputs modified: NO
New files written: NO


In [ ]:

# ============================================================
#RIDGE STACKING VS EGARCH-t + INDIA VIX
# Paired HAC inference on the final OOS evaluation period.
# Read-only: no files written.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ------------------------------------------------------------
# 1. Retrieve the existing VIX-enhanced benchmark
# ------------------------------------------------------------

assert "EGARCH_t_VIX" in stack_test_source.columns, (
    "EGARCH_t_VIX is absent from the frozen OOS forecast file."
)

vix_benchmark = (
    stack_test_source[
        ["Date", "Actual_Volatility", "EGARCH_t_VIX"]
    ]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

ridge_eval = stacking_predictions[
    ["Date", "Actual_Volatility", "Ridge_Stacking"]
].copy()

# Exact date and target alignment is essential for paired tests.
assert len(vix_benchmark) == len(ridge_eval) == 673
assert vix_benchmark["Date"].equals(ridge_eval["Date"])
assert np.allclose(
    vix_benchmark["Actual_Volatility"].to_numpy(),
    ridge_eval["Actual_Volatility"].to_numpy(),
    rtol=0,
    atol=1e-12,
)

actual = ridge_eval["Actual_Volatility"].to_numpy(dtype=float)
ridge_pred = ridge_eval["Ridge_Stacking"].to_numpy(dtype=float)
vix_pred = vix_benchmark["EGARCH_t_VIX"].to_numpy(dtype=float)

# ------------------------------------------------------------
# 2. Compute paired loss differences
# Positive gain means Ridge has lower loss.
# ------------------------------------------------------------

ridge_abs_loss = np.abs(actual - ridge_pred)
vix_abs_loss = np.abs(actual - vix_pred)

ridge_sq_loss = (actual - ridge_pred) ** 2
vix_sq_loss = (actual - vix_pred) ** 2

absolute_gain = vix_abs_loss - ridge_abs_loss
squared_gain = vix_sq_loss - ridge_sq_loss

# ------------------------------------------------------------
# 3. HAC inference with five lags
# ------------------------------------------------------------

def hac_mean_test(loss_gain, maxlags=5):
    fitted = sm.OLS(
        np.asarray(loss_gain, dtype=float),
        np.ones((len(loss_gain), 1)),
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags},
        use_t=True,
    )

    return {
        "Mean_Loss_Gain": float(np.mean(loss_gain)),
        "HAC_Statistic": float(fitted.tvalues[0]),
        "P_Value": float(fitted.pvalues[0]),
    }

abs_test = hac_mean_test(absolute_gain)
sq_test = hac_mean_test(squared_gain)

ridge_mae = float(np.mean(ridge_abs_loss))
vix_mae = float(np.mean(vix_abs_loss))

ridge_rmse = float(np.sqrt(np.mean(ridge_sq_loss)))
vix_rmse = float(np.sqrt(np.mean(vix_sq_loss)))

comparison = pd.DataFrame([
    {
        "Model": "Ridge_Stacking",
        "MAE": ridge_mae,
        "RMSE": ridge_rmse,
    },
    {
        "Model": "EGARCH_t_VIX",
        "MAE": vix_mae,
        "RMSE": vix_rmse,
    },
])

print("=" * 72)
print("RIDGE STACKING VS EGARCH-t + INDIA VIX")
print("=" * 72)

print("Matched OOS observations:", len(actual))
print(
    "Dates:",
    ridge_eval["Date"].min().date(),
    "to",
    ridge_eval["Date"].max().date(),
)

print("\nForecast comparison:")
display(comparison.round(8))

print(
    "\nMAE reduction (%):",
    round(100 * (vix_mae - ridge_mae) / vix_mae, 4),
)
print(
    "RMSE reduction (%):",
    round(100 * (vix_rmse - ridge_rmse) / vix_rmse, 4),
)

print("\nPaired HAC test — absolute-error loss")
display(pd.DataFrame([abs_test]).round(8))

print("\nPaired HAC test — squared-error loss")
display(pd.DataFrame([sq_test]).round(8))

print("\nInterpretation:")
print("Positive mean loss gain favours Ridge stacking.")
print("P-values assess whether the average loss difference differs from zero.")
print("HAC lag: 5")
print("Frozen research outputs modified: NO")
print("New files written: NO")


RIDGE STACKING VS EGARCH-t + INDIA VIX
Matched OOS observations: 673
Dates: 2024-01-01 to 2026-09-22

Forecast comparison:


,Model,MAE,RMSE
0,Ridge_Stacking,0.003700,0.005182
1,EGARCH_t_VIX,0.003737,0.005262



MAE reduction (%): 1.0105
RMSE reduction (%): 1.5233

Paired HAC test — absolute-error loss


,Mean_Loss_Gain,HAC_Statistic,P_Value
0,0.000038,0.420281,0.674415



Paired HAC test — squared-error loss


,Mean_Loss_Gain,HAC_Statistic,P_Value
0,8.400000e-07,0.61716,0.537339



Interpretation:
Positive mean loss gain favours Ridge stacking.
P-values assess whether the average loss difference differs from zero.
HAC lag: 5
Frozen research outputs modified: NO
New files written: NO


# Adaptive Market Risk Intelligence System
## Notebook 04 — ML Volatility Experiments: Work Log, Results, Fixes, and Handoff

**Prepared:** 18 October 2026  
**Notebook reviewed:** `04_ml_volatility_experiments(1).ipynb` (25 cells in the supplied copy)  
**Project root:** `D:\DS Projects\Adaptive Market Risk Intelligence System`  
**Experiment output directory:** `data/ml_experiments/v1/`

> This document records work and outputs evidenced by the supplied Notebook 04 and the results shared during the project conversation. It does not claim that the entire research project has been re-audited. Frozen research artifacts were treated as read-only.

## 1. Purpose

Notebook 04 tests whether machine-learning models, walk-forward ML, simple forecast combinations, and Ridge stacking improve on the established volatility-forecast benchmarks. It is an experimental layer; it does not replace the frozen research forecasts.

The evaluated target is `Actual_Volatility` / `Target_Abs_Return`, representing the absolute one-day log return in the project’s forecast setup. MAE and RMSE measure forecast errors against that target. They should not be described as direct measurements of latent volatility or multi-day realized volatility.

## 2. Data and chronology

### Final out-of-sample forecasts

Frozen source: `data/research_outputs/v1/oos_forecasts.csv`

- **673 observations**, 2024-01-01 to 2026-09-22.
- Forecasts include NIFTY-only baselines `Historical_20D`, `EWMA`, `GARCH_t`, `GJR_GARCH_t`, `EGARCH_t` and VIX-enhanced forecasts `Historical_20D_VIX`, `EWMA_VIX`, `EGARCH_t_VIX`.
- `data/business_inputs/v1/oos_forecasts.csv` is a copy of this 673-row final OOS data, not a separate historical sample.

### Historical validation forecasts for stacking

Existing source: `data/ml_experiments/v1/historical_validation_base_forecasts.csv`

- **493 observations**, 2022-01-03 to 2023-12-29.
- Columns: `Date`, `Actual_Volatility`, `Historical_20D`, `EWMA`, `GARCH_t`, `GJR_GARCH_t`, `EGARCH_t`.
- Read-only integrity check passed: shape `(493, 7)`, expected date range, **0 missing values**.

The historical stacking training dates precede the final OOS evaluation dates; the two periods do not overlap.

## 3. Initial ML experiment (Cells 1–11)

Features recorded in the notebook include:

- `Abs_Return`, `Abs_Return_Lag1`, `Abs_Return_Lag2`, `Abs_Return_Lag3`
- `Mean_Abs_Return_5D`
- `Realized_Volatility_5D`, `Realized_Volatility_20D`
- `EWMA_Volatility_20D`
- `India_VIX_Feature`, `VIX_Change_1D`

Chronological split:

- Training: 2018-02-01 to 2023-12-29, **1,456 observations**.
- Final OOS evaluation: 2024-01-01 to 2026-09-22, **673 observations**.
- The notebook reported training/evaluation separation as **PASS**.

Models included a median `DummyRegressor`, `RandomForestRegressor` (400 trees, `min_samples_leaf=10`, `max_features=0.8`, fixed seed), and `HistGradientBoostingRegressor` (200 iterations, learning rate 0.05, 15 max leaf nodes, L2 regularization 1.0). The model comparison and predictions were saved under `data/ml_experiments/v1/` as `initial_model_comparison.csv` and `initial_model_predictions.csv`. The notebook explicitly reported that frozen research outputs were not modified.

### Initial final-OOS comparison

Lower MAE/RMSE is better; each row has 673 observations.

| Model | MAE | RMSE |
|---|---:|---:|
| EGARCH_t_VIX | 0.003737 | 0.005262 |
| Historical_20D_VIX | 0.003863 | 0.005630 |
| EWMA_VIX | 0.003863 | 0.005637 |
| ML_Random_Forest | 0.004013 | 0.005685 |
| ML_Hist_Gradient_Boosting | 0.004052 | 0.005767 |
| ML_Median_Baseline | 0.004075 | 0.005979 |
| EGARCH_t | 0.004628 | 0.005796 |
| GARCH_t | 0.004852 | 0.006291 |
| GJR_GARCH_t | 0.004865 | 0.006349 |
| EWMA | 0.004924 | 0.006419 |
| Historical_20D | 0.004923 | 0.006564 |

**Finding:** The initial ML models did not outperform the leading `EGARCH_t_VIX` benchmark on this OOS sample.

## 4. Random Forest error and regime analysis (Cell 12)

The saved prediction file was used for read-only paired-error and descriptive-regime analysis.

- RF had lower absolute error than EGARCH-t + VIX on **276** of 673 observations.
- EGARCH-t + VIX had lower absolute error on **397** observations; ties: 0.
- Mean `RF absolute error − EGARCH absolute error`: **0.000275**, favouring EGARCH-t + VIX on average.

| Regime | Model | N | MAE | RMSE |
|---|---|---:|---:|---:|
| Lower | EGARCH_t_VIX | 225 | 0.004163 | 0.004541 |
| Lower | ML_Random_Forest | 225 | 0.004549 | 0.004837 |
| Middle | EGARCH_t_VIX | 224 | 0.001645 | 0.002095 |
| Middle | ML_Random_Forest | 224 | 0.001751 | 0.002242 |
| Higher | EGARCH_t_VIX | 224 | 0.005403 | 0.007621 |
| Higher | ML_Random_Forest | 224 | 0.005736 | 0.008282 |

Outputs saved: `rf_vs_egarch_vix_regime_analysis.csv` and `rf_vs_egarch_vix_error_comparison.csv`. The notebook reported no changes to frozen research outputs or trading thresholds.

## 5. Monthly expanding-window walk-forward ML (Cell 13)

Random Forest and HistGradientBoosting were retrained on an expanding history at monthly evaluation steps. The printed schedule shows that each month’s training cutoff precedes that month’s evaluation dates. Forecast outputs and comparison were saved as:

- `walk_forward_predictions.csv`
- `walk_forward_model_comparison.csv`

### Walk-forward results

| Model | MAE | RMSE | N |
|---|---:|---:|---:|
| EGARCH_t_VIX | 0.003737 | 0.005262 | 673 |
| Historical_20D_VIX | 0.003863 | 0.005630 | 673 |
| EWMA_VIX | 0.003863 | 0.005637 | 673 |
| ML_Random_Forest_Fixed | 0.004013 | 0.005685 | 673 |
| ML_Random_Forest_Walk_Forward | 0.004012 | 0.005687 | 673 |
| ML_Hist_Gradient_Boosting_Fixed | 0.004052 | 0.005767 | 673 |
| EGARCH_t | 0.004628 | 0.005796 | 673 |
| ML_Hist_Gradient_Boosting_Walk_Forward | 0.004105 | 0.005920 | 673 |
| GARCH_t | 0.004852 | 0.006291 | 673 |
| GJR_GARCH_t | 0.004865 | 0.006349 | 673 |
| EWMA | 0.004924 | 0.006419 | 673 |
| Historical_20D | 0.004923 | 0.006564 | 673 |

**Finding:** Walk-forward retraining did not materially improve RF or HGB results and neither beat the leading VIX-enhanced benchmark.

## 6. HAC-adjusted comparison of ML vs EGARCH-t + VIX (Cell 14)

The notebook performed HAC-adjusted tests on absolute-error and squared-error differentials for the fixed RF, walk-forward RF, and walk-forward HGB models, with positive differences defined as higher loss for ML. The saved notebook output should be retained as the authoritative record for exact test statistics and p-values; the overall error metrics show all three ML variants had worse MAE/RMSE than `EGARCH_t_VIX`.

## 7. Forecast mixtures (Cell 15)

Three combinations were evaluated:

- `Mix_Strategy_NIFTY_Only`: equal-weight mean of five NIFTY-only statistical forecasts.
- `Mix_Strategy_All_Forecasts`: equal-weight mean of all eight statistical forecasts.
- `Mix_Strategy_Group_Balanced`: 50% NIFTY-only group mean + 50% VIX-enhanced group mean.

Reported results:

| Model / strategy | MAE | RMSE |
|---|---:|---:|
| EGARCH_t_VIX | 0.003737 | 0.005262 |
| Mix_Strategy_Group_Balanced | 0.004119 | 0.005584 |
| Mix_Strategy_All_Forecasts | 0.004252 | 0.005684 |
| Mix_Strategy_NIFTY_Only / Equal_Weight_Mean | 0.004767 | 0.006140 |
| Equal_Weight_Median | 0.004733 | 0.006136 |

**Finding:** None of the mixtures beat `EGARCH_t_VIX` on MAE. The group-balanced mixture had RMSE 0.005584, still above the benchmark RMSE 0.005262.

## 8. Artifact inventory and source review (Cells 17–18)

The artifact inventory confirmed the relevant final forecast files had 673 observations in the 2024–2026 date range. Source cells in Notebook 02 were inspected without running Notebook 02 or regenerating frozen outputs.

## 9. Historical validation loader and kernel-state fix (Cells 20–21 in notebook headings)

A previous attempt to use the live variable `validation_forecasts` in Notebook 04 failed because that variable was not present in the active Notebook 04 kernel. Separate notebooks do not share live variables merely because they use the same Python environment.

**Fix:** load the already saved `historical_validation_base_forecasts.csv` instead of relying on a cross-notebook variable or rerunning Notebook 02. The read-only validation passed and did not regenerate forecasts or modify frozen research outputs.

Stacking inputs were then prepared from five NIFTY-only forecasts:
`Historical_20D`, `EWMA`, `GARCH_t`, `GJR_GARCH_t`, `EGARCH_t`.

- Training: `(493, 7)`, 2022-01-03 to 2023-12-29.
- Final OOS: `(673, 7)`, 2024-01-01 to 2026-09-22.
- Training/OOS overlap: **NO**.
- Frozen research outputs modified: **NO**.

## 10. Ridge stacking experiment (Cell 22)

Pipeline: `StandardScaler` + `Ridge`. Hyperparameter search used chronological `TimeSeriesSplit(n_splits=5)` and alpha candidates `[0.01, 0.1, 1, 10, 100]`, selecting by negative MAE. The selected model was refit on all 493 historical validation rows and evaluated on the final 673 OOS rows. Predictions were clipped at zero.

- Selected alpha: **10.0**.
- Best chronological CV MAE: **0.00357537**.
- Final OOS MAE: **0.003700**.
- Final OOS RMSE: **0.005182**.
- Mean forecast error: **0.000426**.

Standardized coefficients:

| Feature | Coefficient |
|---|---:|
| Historical_20D | -0.001331 |
| EWMA | 0.001870 |
| GARCH_t | -0.001568 |
| GJR_GARCH_t | -0.002691 |
| EGARCH_t | 0.006307 |

No files were written by the Ridge cell; frozen research outputs were not changed.

## 11. Ridge vs EGARCH-t + VIX: paired HAC comparison (Cell 23)

Both models were compared on the same 673 OOS dates and actual target values. HAC tests used `maxlags=5` and `use_t=True`.

| Model | MAE | RMSE |
|---|---:|---:|
| Ridge_Stacking | **0.003700** | **0.005182** |
| EGARCH_t_VIX | 0.003737 | 0.005262 |

Ridge’s observed reductions relative to EGARCH-t + VIX:

- MAE reduction: **1.0105%**.
- RMSE reduction: **1.5233%**.

| Loss differential (benchmark loss − Ridge loss) | Mean gain | HAC statistic | p-value |
|---|---:|---:|---:|
| Absolute error | 0.000038 | 0.420281 | 0.674415 |
| Squared error | approximately 0.00000084 | 0.617160 | 0.537339 |

**Interpretation:** Ridge had the lowest observed error in this comparison, but neither test rejects equal average loss at 5%. Do not claim that Ridge is statistically superior. Failure to reject is not proof of equivalence. No output files were written.

## 12. Consolidated model results

The following consolidated comparison is from results recorded during this project work. Lower MAE/RMSE is better; 673 OOS observations for the final test-period results.

| Model / strategy | MAE | RMSE |
|---|---:|---:|
| **Ridge_Stacking** | **0.003700** | **0.005182** |
| EGARCH_t_VIX | 0.003737 | 0.005262 |
| Historical_20D_VIX | 0.003863 | 0.005630 |
| EWMA_VIX | 0.003863 | 0.005637 |
| ML_Random_Forest | 0.004013 | 0.005685 |
| ML_Random_Forest_WalkForward | 0.004012 | 0.005687 |
| ML_Hist_Gradient_Boosting | 0.004052 | 0.005767 |
| ML_Median_Baseline | 0.004075 | 0.005979 |
| ML_Hist_Gradient_Boosting_WalkForward | 0.004105 | 0.005920 |
| Mix_Strategy_Group_Balanced | 0.004119 | 0.005584 |
| Mix_Strategy_All_Forecasts | 0.004252 | 0.005684 |
| EGARCH_t | 0.004628 | 0.005796 |
| Equal_Weight_Median | 0.004733 | 0.006136 |
| Equal_Weight_Mean / Mix_Strategy_NIFTY_Only | 0.004767 | 0.006140 |
| GARCH_t | 0.004852 | 0.006291 |
| GJR_GARCH_t | 0.004865 | 0.006349 |
| EWMA | 0.004924 | 0.006419 |
| Historical_20D | 0.004923 | 0.006564 |

Earlier paired comparisons also reported improvements from adding India VIX:

| Comparison | MAE reduction | RMSE reduction | HAC p-value for absolute-loss gain |
|---|---:|---:|---:|
| Historical_20D → Historical_20D_VIX | 21.50% | 14.20% | 8.4 × 10⁻⁷ |
| EWMA → EWMA_VIX | 21.55% | 12.19% | Printed as 0.000000 |
| EGARCH_t → EGARCH_t_VIX | 19.23% | 9.23% | Printed as 0.000000 |

`0.000000` is a rounded display, not a literal p-value of zero. The established VIX-enhanced benchmark remains `EGARCH_t_VIX`; Ridge’s small numerical advantage is exploratory because paired tests were not significant.

## 13. Targeted forecast-construction consistency check

A source search in Notebook 02 found one-step-ahead forecasting procedures and rules requiring information to be available before each forecast date. It also surfaced strict walk-forward calibration and timing checks. However, the search output contained truncated source cells and did not conclusively prove that all five base forecasts use perfectly matched estimation, initialization, recursion, and refitting rules across the 2022–2023 stacking-training period and the 2024–2026 final OOS period.

This limitation does not invalidate the descriptive results on the supplied files, but it is relevant if Ridge is ever made a central research claim. No full audit or rerun was performed; the decision was to stop at a targeted check and preserve frozen outputs.

## 14. Fixes and safeguards established

1. Removed the dependency on a live variable from another notebook by loading the saved historical validation CSV.
2. Verified that the 493-row historical stacking set and 673-row final OOS set have no date overlap.
3. Kept `data/research_outputs/v1/oos_forecasts.csv` frozen; no stacking cell regenerated or overwrote it.
4. Reported paired HAC p-values alongside Ridge’s small observed advantage.
5. Avoided unnecessary environment reinstalls after sklearn and plotting tests subsequently succeeded.
6. Kept the forecast-generation audit targeted; did not rerun Notebook 02.

## 15. Remaining limitations

- No complete cell-by-cell comparison of all five base forecast-generation procedures across both time periods.
- Ridge has not been promoted to the primary research model.
- The Ridge results were displayed in notebook output; the Ridge cells did not save a new results file.
- VaR / Expected Shortfall robustness and backtesting are a separate phase not completed by this notebook.

## 16. Recommended next step

Do not add more ML models yet. Begin a separate VaR / Expected Shortfall robustness experiment:

1. Inspect the existing VaR/ES definitions and backtesting sections first.
2. Preserve the existing baseline outputs.
3. Compare rolling windows of 126, 252, and 500 trading days where the data support them.
4. Evaluate VaR exceedances and ES behavior using clearly stated confidence levels and appropriate backtesting methods.
5. Save new outputs to a separate experiment directory, not `research_outputs/v1`.
6. Document results and limitations before using them in the paper or dashboard.

Continue the agreed workflow: **one cell at a time, review each output before proceeding, do not rerun completed cells, and do not alter frozen outputs without a specific reason.**

## 17. Handoff summary

Notebook 04 tested ML baselines, fixed and monthly walk-forward ML, forecast mixtures, and Ridge stacking against established statistical forecasts. The 673-observation final OOS period is 2024-01-01 to 2026-09-22; the historical stacking-training artifact has 493 observations from 2022-01-03 to 2023-12-29. A kernel-state issue was resolved by reading the saved validation CSV rather than relying on a variable in another notebook’s kernel. Ridge selected alpha 10 and achieved MAE 0.003700 / RMSE 0.005182, slightly below EGARCH-t + VIX at 0.003737 / 0.005262, but paired HAC p-values (0.6744 absolute loss; 0.5373 squared loss) do not establish significant superiority. Frozen research outputs were preserved. Next phase: inspect existing VaR/ES code, then perform a separate rolling-window robustness experiment.